# Robust observation models

外れ値の生成機構に応じて Student-t と contamination mixture を使い分けます。Relevance Pursuit は `10_robust_gp.ipynb` を参照してください。


## 準備


In [ ]:
import torch
from robotorchan.models import StudentTSingleTaskGP, ContaminatedSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 16).unsqueeze(-1)
Y = torch.sin(2 * torch.pi * X)
Y[7] += 2.0


## Student-t likelihood


In [ ]:
student = StudentTSingleTaskGP(X, Y, num_inducing=8, df=4.0)
optimizer = torch.optim.Adam(student.parameters(), lr=0.03)
for _ in range(3):
    optimizer.zero_grad()
    loss = student.training_loss()
    loss.backward()
    optimizer.step()
print(student.posterior(torch.tensor([[0.25], [0.75]])).mean)


## Contamination mixture


In [ ]:
contaminated = ContaminatedSingleTaskGP(
    X, Y, contamination_probability=0.1, num_inducing=8, num_likelihood_samples=4
)
print(contaminated.training_loss(num_likelihood_samples=4).item())
print(contaminated.contamination_diagnostic(X, Y).squeeze(-1))
